# 03 -- Probability Calibration

**Contact Luck Prototype v0.1**

Why calibration matters more here than plain ranking accuracy: the luck score is built from the *magnitude* of predicted probabilities (`expected_value = sum(p(outcome) * value(outcome))`), not just which outcome is most likely. A model that ranks outcomes correctly but assigns systematically over- or under-confident probabilities will produce biased raw-luck values even when its classification-style metrics look fine.

Running this notebook successfully is **not** evidence that the model is well-calibrated -- that must be judged from the table/plots below, on real held-out (non-2025) data.

> **This repository is a research prototype, not a validated public baseball statistic.** See `README.md` and `CLAUDE.md` for full scope and limitations.

In [ ]:
import pandas as pd

from mlb_luck_score.config import FIGURES_DIR, PROCESSED_DATA_DIR, TRAIN_SEASONS, VALIDATION_SEASONS
from mlb_luck_score.models.calibrate_model import compute_calibration_table, plot_calibration_curves
from mlb_luck_score.models.train_contact_model import predict_proba_ordered, train_model

CLEANED_PATH = PROCESSED_DATA_DIR / "cleaned_batted_balls.parquet"
pd.set_option("display.width", 120)

In [ ]:
if CLEANED_PATH.exists():
    df = pd.read_parquet(CLEANED_PATH)
    training_eligible = df[df["eligible_for_training"].astype(bool)]
    train_df = training_eligible[training_eligible["season"].isin(TRAIN_SEASONS)]
    val_df = training_eligible[training_eligible["season"].isin(VALIDATION_SEASONS)]
else:
    train_df = val_df = None
    print(
        f"No cleaned data found at {CLEANED_PATH}.\n"
        "Run `make download-sample` then `make clean-data`, then re-run this notebook."
    )

## Calibration table

In [ ]:
if train_df is not None and len(train_df) > 0 and val_df is not None and len(val_df) > 0:
    trained = train_model(train_df)
    feature_cols = trained.numeric_features + trained.categorical_features
    proba_df = predict_proba_ordered(trained, val_df[feature_cols])
    calibration_table = compute_calibration_table(val_df["outcome_class"], proba_df)
    display(calibration_table)
else:
    calibration_table = None
    print("Skipped -- need both training and validation rows to compute calibration.")

## One plot per outcome class

In [ ]:
if calibration_table is not None and not calibration_table.empty:
    paths = plot_calibration_curves(calibration_table, FIGURES_DIR)
    print("Saved:", paths)
    from IPython.display import Image, display as ipy_display

    for path in paths:
        ipy_display(Image(filename=str(path)))
else:
    print("Skipped -- no calibration table available.")

## Interpretation warnings

- Bins with very few samples are marked `reliable=False` in the table above -- treat their calibration error as noisy, not meaningful.
- A small one-week bootstrap sample will generally NOT produce a reliable calibration curve. Do not draw scientific conclusions from it.
- Calibration should be re-checked whenever the feature set, training window, or model type changes.

In [ ]:
if calibration_table is not None and not calibration_table.empty:
    print("Sample-size summary by outcome class:")
    display(calibration_table.groupby("outcome_class")["sample_count"].sum())
    n_unreliable = int((~calibration_table["reliable"]).sum())
    msg = f"{n_unreliable} of {len(calibration_table)} bins are marked unreliable"
    print(msg + " (fewer than the minimum sample threshold).")
else:
    print("Skipped -- no calibration table available.")